In [ ]:
%%sql -r ai_agent_activity
with prod_tenant_info as (

    select
        tenant_id,
        tenant_nm,
        account_id,
        data_cntr_cd,
        regexp_replace(data_cntr_cd, '[0-9]+$', '') as region
    from PROD_HARMONIZED.PRODUCT.TENANT_PROF
    where crnt_rcrd_ind = 'Y'
      and tenant_typ_nm = 'PRODUCTION'
 --     and organization_typ_nm = 'CUSTOMER'
      and tenant_enbld_status_nm = 'ENABLED'
),

api_activity as (

    select
        data_cntr_cd,
        tenant_id,
        user_id,
        concat(data_cntr_cd, '.', tenant_id) as vh_tenant_id,
        request_endpoint_nm,
        request_method_cd,
        request_start_dts,
        request_end_dts,
        client_context_hdr_nm,
        user_agent
    from PROD_HARMONIZED.PRODUCT.USER_API_ACTIVITY
    where request_end_dts >= '2025-05-01'::timestamp
    --  and user_imprsntd_ind = 'N'
   --   and request_status_cd = '200'
      and (
            request_endpoint_nm like '/api/ai/topics/{p}/conversations/{p}/chat/%'
            or (request_method_cd = 'GET' and request_endpoint_nm = '/api/ai/topics/{p}/conversations/{p}/messages/{p}/templateAdhoc/')
          )
)

    select 
        region,
        api.vh_tenant_id,
        pti.tenant_nm,
        api.user_id,
        api.request_start_dts as action_timestamp_start,
        api.request_end_dts as action_timestamp_end,
        --agent categorization
        case 
            when api.request_method_cd = 'GET' and api.request_endpoint_nm = '/api/ai/topics/{p}/conversations/{p}/messages/{p}/templateAdhoc/' then 'reporting agent'
            when (api.client_context_hdr_nm ilike '%mql%' or api.client_context_hdr_nm ilike '%query%')
                 and api.request_endpoint_nm = '/api/ai/topics/{p}/conversations/{p}/chat/' then 'MQL agent prompt'
            when api.client_context_hdr_nm ilike '%planning_agent%'
                 and api.request_endpoint_nm = '/api/ai/topics/{p}/conversations/{p}/chat/' then 'Planning agent prompt'
            else 'insights agent' 
        end as agent_type,
    from api_activity api
    inner join prod_tenant_info pti 
        on pti.tenant_id = api.tenant_id 
       and pti.data_cntr_cd = api.data_cntr_cd

union all 

select
prod_tenant_info.region,
concat(prod_tenant_info.data_cntr_cd,'.',customer_id) as vh_tenant_id, 
tenant_nm,
staging.user_id,
min(action_datetime) as action_timestamp,
dateadd('second', 30, min(action_datetime)) as action_timestamp_end,
'ad hoc - reporting agent proxy' as agent_type 
from PROD_RAW.PROD_TEAM_SANDBOX.ADHOC_RUM_DATA_STAGING staging 
inner join prod_tenant_info on concat(staging.region,'.',customer_id) = concat(prod_tenant_info.data_cntr_cd,'.',prod_tenant_info.tenant_id)
where action_datetime >= '2025-05-01'::timestamp
group by prod_tenant_info.region, concat(prod_tenant_info.data_cntr_cd,'.',customer_id), tenant_nm, staging.user_id, file_id

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

region_counts = ai_agent_activity.groupby('REGION').size()

fig, ax = plt.subplots(figsize=(8, 6))
ax.pie(
    region_counts.values,
    labels=region_counts.index,
    autopct='%1.1f%%',
    startangle=140
)
ax.set_title('Rows per Region')
plt.tight_layout()
plt.show()